# Capstone 3 — RAG-Powered Customer Support Agent

**Book:** [Agentic AI: Build, Ship, Portfolio](../index.html)

---

In this capstone you will build a **RAG-powered customer support agent** that:

1. Builds a searchable knowledge base from support documents using ChromaDB.
2. Understands customer queries via intent classification and entity extraction.
3. Retrieves relevant knowledge articles using RAG.
4. Generates accurate, grounded responses.
5. Detects customer sentiment and adjusts tone.
6. Escalates to a human when confidence is low or sentiment is negative.
7. Manages support tickets through their lifecycle.

> **Note:** This notebook uses *mock/simulated* LLM and vector store responses so it runs without an API key or ChromaDB installation. Replace the mocks with real implementations for production use.

---
## 1 Setup

In [ ]:
# --- Setup ---
# %pip install chromadb  # Uncomment for real ChromaDB usage

import json
import hashlib
import re
import textwrap
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any

print("All imports ready.")

In [ ]:
# --- Mock LLM Helper ---

def mock_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """Simulate an LLM response based on keywords in the prompt."""
    lower = (system_prompt + user_prompt).lower()
    if "intent" in lower and ("classif" in lower or "detect" in lower):
        return json.dumps({
            "intent": "billing_inquiry",
            "confidence": 0.87,
            "entities": [
                {"type": "product", "value": "Pro Plan"},
                {"type": "issue", "value": "overcharge"}
            ],
            "sub_intent": "refund_request"
        })
    elif "sentiment" in lower:
        return json.dumps({
            "sentiment": "frustrated",
            "score": -0.6,
            "indicators": ["been waiting", "still not resolved", "very disappointed"]
        })
    elif "response" in lower or "answer" in lower or "reply" in lower:
        return ("I understand your concern about the billing issue with your Pro Plan. "
                "I can see that the charge in question was processed on your last billing cycle. "
                "Here is what I can do to help:\n\n"
                "1. I will review your account for any billing discrepancies.\n"
                "2. If an overcharge is confirmed, I will initiate a refund within 3-5 business days.\n"
                "3. I will also ensure your plan is correctly configured going forward.\n\n"
                "Is there anything else I can help you with?")
    elif "escalat" in lower:
        return json.dumps({
            "should_escalate": True,
            "reason": "Customer sentiment is very negative and issue requires manual account review.",
            "priority": "high",
            "suggested_team": "billing_specialists"
        })
    else:
        return "Mock response: Request processed."

print("Mock LLM helper ready.")

---
## 2 Knowledge Base Builder (ChromaDB)

The knowledge base stores support documents as embeddings for semantic search. In production this uses ChromaDB or a similar vector store. Here we simulate the vector store with an in-memory dictionary and keyword matching.

In [ ]:
@dataclass
class KnowledgeArticle:
    """A single knowledge base article."""
    article_id: str
    title: str
    content: str
    category: str
    tags: list[str]
    last_updated: str


@dataclass
class SearchHit:
    """A search result from the knowledge base."""
    article: KnowledgeArticle
    score: float  # 0.0 to 1.0
    matched_chunk: str


class MockVectorStore:
    """Simulates a ChromaDB vector store with keyword-based retrieval."""

    def __init__(self):
        self.articles: dict[str, KnowledgeArticle] = {}

    def add(self, article: KnowledgeArticle) -> None:
        """Add an article to the store (simulates embedding + indexing)."""
        self.articles[article.article_id] = article

    def search(self, query: str, top_k: int = 3) -> list[SearchHit]:
        """Search for relevant articles (simulates semantic search)."""
        query_words = set(query.lower().split())
        scored: list[tuple[float, KnowledgeArticle]] = []

        for article in self.articles.values():
            # Simple keyword overlap scoring
            article_words = set(
                (article.title + " " + article.content + " " + " ".join(article.tags)).lower().split()
            )
            overlap = len(query_words & article_words)
            if overlap > 0:
                score = min(overlap / max(len(query_words), 1), 1.0)
                scored.append((score, article))

        scored.sort(key=lambda x: x[0], reverse=True)
        results = []
        for score, article in scored[:top_k]:
            results.append(SearchHit(
                article=article,
                score=round(score, 3),
                matched_chunk=article.content[:200]
            ))
        return results

    @property
    def count(self) -> int:
        return len(self.articles)

print("MockVectorStore defined.")

In [ ]:
# --- Build the Knowledge Base ---

SAMPLE_ARTICLES = [
    KnowledgeArticle(
        article_id="KB001", title="Billing FAQ",
        content="Our billing cycle runs on the 1st of each month. Pro Plan costs $49/month. "
                "Enterprise Plan costs $199/month. Refunds are processed within 3-5 business days. "
                "Contact billing@example.com for disputes.",
        category="billing", tags=["billing", "refund", "plan", "pricing", "overcharge"],
        last_updated="2026-01-15"
    ),
    KnowledgeArticle(
        article_id="KB002", title="Account Setup Guide",
        content="To set up your account: 1) Visit app.example.com/signup. 2) Enter your email. "
                "3) Choose a plan (Free, Pro, or Enterprise). 4) Complete payment. "
                "5) Verify your email. Password must be 8+ characters with a number.",
        category="onboarding", tags=["account", "setup", "signup", "password", "onboarding"],
        last_updated="2026-02-01"
    ),
    KnowledgeArticle(
        article_id="KB003", title="API Rate Limits",
        content="Free Plan: 100 requests/day. Pro Plan: 10,000 requests/day. "
                "Enterprise Plan: 100,000 requests/day. Rate limit headers are included "
                "in every response. If you hit the limit, wait or upgrade your plan.",
        category="technical", tags=["api", "rate limit", "technical", "plan", "requests"],
        last_updated="2026-01-20"
    ),
    KnowledgeArticle(
        article_id="KB004", title="Cancellation and Refund Policy",
        content="You can cancel your subscription at any time from Settings > Billing > Cancel. "
                "Refunds are available for the current billing period if requested within 7 days. "
                "Pro-rated refunds are not available. Annual plans can be refunded within 30 days.",
        category="billing", tags=["cancel", "refund", "subscription", "billing", "policy"],
        last_updated="2026-02-10"
    ),
    KnowledgeArticle(
        article_id="KB005", title="Integration Troubleshooting",
        content="Common integration issues: 1) Invalid API key — regenerate from Dashboard. "
                "2) CORS errors — add your domain to the allowlist. 3) Timeout errors — "
                "check your network and our status page. 4) 500 errors — contact support.",
        category="technical", tags=["integration", "api", "troubleshooting", "error", "cors"],
        last_updated="2026-01-25"
    )
]

kb = MockVectorStore()
for article in SAMPLE_ARTICLES:
    kb.add(article)

print(f"Knowledge base built: {kb.count} articles")

In [ ]:
# --- Test the Knowledge Base ---

test_hits = kb.search("billing refund overcharge Pro Plan", top_k=3)
for hit in test_hits:
    print(f"  [{hit.score:.3f}] {hit.article.title} ({hit.article.article_id})")
    print(f"          {hit.matched_chunk[:100]}...")

---
## 3 Query Understanding Agent

The **Query Understanding Agent** classifies the customer's intent, extracts entities, and determines the sub-intent for routing.

In [ ]:
@dataclass
class QueryUnderstanding:
    """Result of understanding a customer query."""
    original_query: str
    intent: str
    sub_intent: str
    confidence: float
    entities: list[dict[str, str]]
    search_query: str  # Reformulated for KB search


class QueryUnderstandingAgent:
    """Classifies intent and extracts entities from customer messages."""

    SUPPORTED_INTENTS = [
        "billing_inquiry", "technical_support", "account_management",
        "feature_request", "complaint", "cancellation", "general_question"
    ]

    def understand(self, query: str) -> QueryUnderstanding:
        """Analyze a customer query."""
        raw = mock_llm(
            system_prompt=(
                "You are a customer intent classifier. "
                f"Supported intents: {self.SUPPORTED_INTENTS}. "
                "Detect intent, confidence, entities, and sub-intent."
            ),
            user_prompt=f"Customer message: {query}"
        )
        data = json.loads(raw)

        # Build optimized search query from entities
        entity_values = [e["value"] for e in data["entities"]]
        search_query = f"{data['intent']} {' '.join(entity_values)}"

        result = QueryUnderstanding(
            original_query=query,
            intent=data["intent"],
            sub_intent=data.get("sub_intent", ""),
            confidence=data["confidence"],
            entities=data["entities"],
            search_query=search_query
        )

        print(f"[QueryUnderstanding] Intent: {result.intent} ({result.confidence:.2f}), "
              f"Entities: {result.entities}")
        return result

print("QueryUnderstandingAgent defined.")

In [ ]:
# --- Test Query Understanding ---

qu_agent = QueryUnderstandingAgent()
understanding = qu_agent.understand(
    "I was overcharged on my Pro Plan last month and I want a refund. "
    "I've been waiting for a week and still no response!"
)

print(f"\nIntent:       {understanding.intent}")
print(f"Sub-intent:   {understanding.sub_intent}")
print(f"Confidence:   {understanding.confidence}")
print(f"Entities:     {understanding.entities}")
print(f"Search query: {understanding.search_query}")

---
## 4 RAG Retrieval Pipeline

The **RAG Pipeline** combines query understanding with knowledge base search to retrieve the most relevant context for response generation.

In [ ]:
@dataclass
class RetrievalResult:
    """Result of the RAG retrieval pipeline."""
    query_understanding: QueryUnderstanding
    hits: list[SearchHit]
    context: str  # Combined context for the response generator
    context_sufficient: bool


class RAGRetrievalPipeline:
    """Combines query understanding with knowledge base retrieval."""

    def __init__(self, kb: MockVectorStore, min_score: float = 0.2, top_k: int = 3):
        self.kb = kb
        self.min_score = min_score
        self.top_k = top_k
        self.qu_agent = QueryUnderstandingAgent()

    def retrieve(self, customer_query: str) -> RetrievalResult:
        """Full retrieval pipeline: understand -> search -> build context."""
        # Step 1: Understand the query
        understanding = self.qu_agent.understand(customer_query)

        # Step 2: Search the knowledge base
        hits = self.kb.search(understanding.search_query, top_k=self.top_k)

        # Step 3: Filter by minimum score
        relevant_hits = [h for h in hits if h.score >= self.min_score]

        # Step 4: Build context string
        context_parts = []
        for h in relevant_hits:
            context_parts.append(
                f"[{h.article.article_id}] {h.article.title}:\n{h.article.content}"
            )
        context = "\n\n---\n\n".join(context_parts) if context_parts else "No relevant articles found."

        result = RetrievalResult(
            query_understanding=understanding,
            hits=relevant_hits,
            context=context,
            context_sufficient=len(relevant_hits) > 0
        )

        print(f"[RAGRetrieval] {len(relevant_hits)} relevant articles found "
              f"(min_score={self.min_score})")
        return result

print("RAGRetrievalPipeline defined.")

In [ ]:
# --- Test RAG Retrieval ---

rag_pipeline = RAGRetrievalPipeline(kb=kb, min_score=0.2, top_k=3)
retrieval = rag_pipeline.retrieve(
    "I was overcharged on my Pro Plan last month and I want a refund."
)

print(f"\nContext sufficient: {retrieval.context_sufficient}")
print(f"Articles retrieved: {len(retrieval.hits)}")
for h in retrieval.hits:
    print(f"  [{h.score:.3f}] {h.article.title}")
print(f"\nContext (first 300 chars):\n{retrieval.context[:300]}...")

---
## 5 Response Generator

The **Response Generator** uses the retrieved context and query understanding to generate a grounded, helpful response.

In [ ]:
@dataclass
class SupportResponse:
    """A generated support response."""
    message: str
    grounded: bool  # Whether the response is backed by KB articles
    sources: list[str]  # Article IDs used
    confidence: float
    suggested_actions: list[str]


class ResponseGenerator:
    """Generates grounded responses using RAG context."""

    def generate(self, retrieval: RetrievalResult,
                 sentiment_adjustment: str | None = None) -> SupportResponse:
        """Generate a response based on retrieval results."""
        tone_instruction = ""
        if sentiment_adjustment == "empathetic":
            tone_instruction = (" The customer is frustrated. Start with empathy "
                                "and acknowledge their experience.")
        elif sentiment_adjustment == "professional":
            tone_instruction = " Keep the tone professional and factual."

        response_text = mock_llm(
            system_prompt=(
                "You are a helpful customer support agent. "
                "Answer based ONLY on the provided context. "
                "If the context does not contain enough information, say so."
                + tone_instruction
            ),
            user_prompt=(
                f"Customer query: {retrieval.query_understanding.original_query}\n\n"
                f"Context:\n{retrieval.context}\n\n"
                f"Generate a helpful response."
            )
        )

        sources = [h.article.article_id for h in retrieval.hits]
        confidence = min(
            retrieval.query_understanding.confidence,
            max((h.score for h in retrieval.hits), default=0.0)
        )

        result = SupportResponse(
            message=response_text,
            grounded=retrieval.context_sufficient,
            sources=sources,
            confidence=round(confidence, 3),
            suggested_actions=["Review account billing", "Process refund if applicable"]
        )

        print(f"[ResponseGenerator] Confidence: {result.confidence:.3f}, "
              f"Grounded: {result.grounded}, Sources: {result.sources}")
        return result

print("ResponseGenerator defined.")

In [ ]:
# --- Test Response Generator ---

response_gen = ResponseGenerator()
response = response_gen.generate(retrieval, sentiment_adjustment="empathetic")

print(f"\nResponse:\n{response.message}")
print(f"\nConfidence: {response.confidence}")
print(f"Grounded:   {response.grounded}")
print(f"Sources:    {response.sources}")

---
## 6 Sentiment Detection

The **Sentiment Detector** analyzes customer messages for emotional tone and provides a score that influences response generation and escalation decisions.

In [ ]:
@dataclass
class SentimentResult:
    """Result of sentiment analysis."""
    sentiment: str  # positive, neutral, frustrated, angry
    score: float  # -1.0 (very negative) to 1.0 (very positive)
    indicators: list[str]
    tone_recommendation: str  # How the response should be toned


class SentimentDetector:
    """Detects customer sentiment and recommends response tone."""

    TONE_MAP = {
        (-1.0, -0.5): ("angry", "empathetic"),
        (-0.5, -0.1): ("frustrated", "empathetic"),
        (-0.1, 0.3): ("neutral", "professional"),
        (0.3, 1.0): ("positive", "friendly")
    }

    def detect(self, message: str) -> SentimentResult:
        """Analyze sentiment of a customer message."""
        raw = mock_llm(
            system_prompt="You are a sentiment analyzer for customer support messages.",
            user_prompt=f"Analyze sentiment: {message}"
        )
        data = json.loads(raw)

        score = data["score"]
        tone_rec = "professional"
        for (low, high), (_, tone) in self.TONE_MAP.items():
            if low <= score <= high:
                tone_rec = tone
                break

        result = SentimentResult(
            sentiment=data["sentiment"],
            score=score,
            indicators=data["indicators"],
            tone_recommendation=tone_rec
        )

        print(f"[Sentiment] {result.sentiment} (score={result.score:.2f}), "
              f"Tone: {result.tone_recommendation}")
        return result

print("SentimentDetector defined.")

In [ ]:
# --- Test Sentiment Detection ---

sentiment_detector = SentimentDetector()
sentiment = sentiment_detector.detect(
    "I've been waiting for a week and this is still not resolved. Very disappointed!"
)

print(f"\nSentiment:  {sentiment.sentiment}")
print(f"Score:      {sentiment.score}")
print(f"Indicators: {sentiment.indicators}")
print(f"Tone rec:   {sentiment.tone_recommendation}")

---
## 7 Escalation Logic

The **Escalation Engine** decides when a conversation should be handed off to a human agent, based on sentiment, confidence, topic complexity, and conversation history.

In [ ]:
@dataclass
class EscalationDecision:
    """Escalation decision."""
    should_escalate: bool
    reason: str
    priority: str  # low, medium, high, urgent
    suggested_team: str
    context_summary: str


class EscalationEngine:
    """Decides when to escalate to a human agent."""

    def __init__(self, sentiment_threshold: float = -0.5,
                 confidence_threshold: float = 0.5,
                 max_turns_before_escalation: int = 5):
        self.sentiment_threshold = sentiment_threshold
        self.confidence_threshold = confidence_threshold
        self.max_turns = max_turns_before_escalation

    # Intent -> team routing
    TEAM_ROUTING = {
        "billing_inquiry": "billing_specialists",
        "technical_support": "technical_team",
        "complaint": "customer_success",
        "cancellation": "retention_team",
    }

    def evaluate(self, sentiment: SentimentResult,
                 response: SupportResponse,
                 understanding: QueryUnderstanding,
                 turn_count: int = 1) -> EscalationDecision:
        """Evaluate whether to escalate."""
        reasons = []

        # Rule 1: Very negative sentiment
        if sentiment.score <= self.sentiment_threshold:
            reasons.append(f"Negative sentiment (score={sentiment.score:.2f})")

        # Rule 2: Low confidence
        if response.confidence < self.confidence_threshold:
            reasons.append(f"Low confidence (score={response.confidence:.2f})")

        # Rule 3: No grounded context
        if not response.grounded:
            reasons.append("Response not grounded in knowledge base")

        # Rule 4: Too many turns
        if turn_count >= self.max_turns:
            reasons.append(f"Conversation exceeded {self.max_turns} turns")

        should_escalate = len(reasons) > 0
        priority = "low"
        if len(reasons) >= 3:
            priority = "urgent"
        elif len(reasons) >= 2:
            priority = "high"
        elif reasons:
            priority = "medium"

        team = self.TEAM_ROUTING.get(understanding.intent, "general_support")

        decision = EscalationDecision(
            should_escalate=should_escalate,
            reason="; ".join(reasons) if reasons else "No escalation needed",
            priority=priority,
            suggested_team=team,
            context_summary=(
                f"Intent: {understanding.intent}, "
                f"Sentiment: {sentiment.sentiment}, "
                f"Turn: {turn_count}"
            )
        )

        print(f"[Escalation] Escalate={decision.should_escalate}, "
              f"Priority={decision.priority}, Team={decision.suggested_team}")
        return decision

print("EscalationEngine defined.")

In [ ]:
# --- Test Escalation Logic ---

escalation_engine = EscalationEngine()
escalation = escalation_engine.evaluate(
    sentiment=sentiment,
    response=response,
    understanding=understanding,
    turn_count=2
)

print(f"\nShould escalate: {escalation.should_escalate}")
print(f"Priority:        {escalation.priority}")
print(f"Reason:          {escalation.reason}")
print(f"Team:            {escalation.suggested_team}")

---
## 8 Ticket Management

The **Ticket Manager** tracks support conversations through their lifecycle: open, in-progress, waiting-on-customer, escalated, resolved, closed.

In [ ]:
class TicketStatus(Enum):
    OPEN = "open"
    IN_PROGRESS = "in_progress"
    WAITING_ON_CUSTOMER = "waiting_on_customer"
    ESCALATED = "escalated"
    RESOLVED = "resolved"
    CLOSED = "closed"


@dataclass
class ConversationTurn:
    """A single turn in the conversation."""
    role: str  # customer, agent, system
    message: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class SupportTicket:
    """A support ticket."""
    ticket_id: str
    customer_id: str
    status: TicketStatus
    intent: str
    priority: str
    conversation: list[ConversationTurn]
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.now().isoformat())
    assigned_team: str = "auto"
    resolution: str = ""


class TicketManager:
    """Manages support tickets."""

    def __init__(self):
        self.tickets: dict[str, SupportTicket] = {}

    def create_ticket(self, customer_id: str, message: str,
                      understanding: QueryUnderstanding) -> SupportTicket:
        """Create a new support ticket."""
        ticket_id = f"TKT-{uuid.uuid4().hex[:8].upper()}"
        ticket = SupportTicket(
            ticket_id=ticket_id,
            customer_id=customer_id,
            status=TicketStatus.OPEN,
            intent=understanding.intent,
            priority="medium",
            conversation=[ConversationTurn(role="customer", message=message)]
        )
        self.tickets[ticket_id] = ticket
        print(f"[TicketManager] Created {ticket_id} for customer {customer_id}")
        return ticket

    def add_agent_response(self, ticket_id: str, response: SupportResponse) -> None:
        """Add an agent response to a ticket."""
        ticket = self.tickets[ticket_id]
        ticket.conversation.append(
            ConversationTurn(
                role="agent", message=response.message,
                metadata={"confidence": response.confidence, "sources": response.sources}
            )
        )
        ticket.status = TicketStatus.WAITING_ON_CUSTOMER
        ticket.updated_at = datetime.now().isoformat()

    def escalate(self, ticket_id: str, decision: EscalationDecision) -> None:
        """Escalate a ticket to a human team."""
        ticket = self.tickets[ticket_id]
        ticket.status = TicketStatus.ESCALATED
        ticket.priority = decision.priority
        ticket.assigned_team = decision.suggested_team
        ticket.conversation.append(
            ConversationTurn(
                role="system",
                message=f"Escalated to {decision.suggested_team}: {decision.reason}"
            )
        )
        ticket.updated_at = datetime.now().isoformat()
        print(f"[TicketManager] {ticket_id} escalated to {decision.suggested_team}")

    def resolve(self, ticket_id: str, resolution: str) -> None:
        """Mark a ticket as resolved."""
        ticket = self.tickets[ticket_id]
        ticket.status = TicketStatus.RESOLVED
        ticket.resolution = resolution
        ticket.updated_at = datetime.now().isoformat()

    def get_summary(self, ticket_id: str) -> str:
        """Get a summary of the ticket."""
        t = self.tickets[ticket_id]
        return (f"Ticket {t.ticket_id} | Status: {t.status.value} | "
                f"Intent: {t.intent} | Priority: {t.priority} | "
                f"Turns: {len(t.conversation)} | Team: {t.assigned_team}")

print("TicketManager defined.")

In [ ]:
# --- Test Ticket Management ---

ticket_mgr = TicketManager()
ticket = ticket_mgr.create_ticket(
    customer_id="CUST-001",
    message="I was overcharged on my Pro Plan!",
    understanding=understanding
)
ticket_mgr.add_agent_response(ticket.ticket_id, response)
print(ticket_mgr.get_summary(ticket.ticket_id))

# Simulate escalation
ticket_mgr.escalate(ticket.ticket_id, escalation)
print(ticket_mgr.get_summary(ticket.ticket_id))

---
## 9 Full Support Pipeline

The complete pipeline ties all components together: query understanding, retrieval, sentiment detection, response generation, escalation, and ticket management.

In [ ]:
class SupportPipeline:
    """Full customer support pipeline."""

    def __init__(self, kb: MockVectorStore):
        self.rag = RAGRetrievalPipeline(kb=kb)
        self.sentiment_detector = SentimentDetector()
        self.response_gen = ResponseGenerator()
        self.escalation_engine = EscalationEngine()
        self.ticket_mgr = TicketManager()

    def handle(self, customer_id: str, message: str,
               verbose: bool = True) -> dict[str, Any]:
        """Handle a customer message through the full pipeline."""
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Customer: {customer_id}")
            print(f"  Message: {message[:80]}...")
            print(f"{'='*60}")

        # Step 1: Retrieve (includes query understanding)
        if verbose: print("\n--- Step 1: Retrieval ---")
        retrieval = self.rag.retrieve(message)

        # Step 2: Sentiment
        if verbose: print("\n--- Step 2: Sentiment ---")
        sentiment = self.sentiment_detector.detect(message)

        # Step 3: Generate Response
        if verbose: print("\n--- Step 3: Response ---")
        response = self.response_gen.generate(
            retrieval, sentiment_adjustment=sentiment.tone_recommendation
        )

        # Step 4: Create Ticket
        if verbose: print("\n--- Step 4: Ticket ---")
        ticket = self.ticket_mgr.create_ticket(
            customer_id, message, retrieval.query_understanding
        )
        self.ticket_mgr.add_agent_response(ticket.ticket_id, response)

        # Step 5: Escalation Check
        if verbose: print("\n--- Step 5: Escalation Check ---")
        escalation = self.escalation_engine.evaluate(
            sentiment=sentiment,
            response=response,
            understanding=retrieval.query_understanding,
            turn_count=1
        )
        if escalation.should_escalate:
            self.ticket_mgr.escalate(ticket.ticket_id, escalation)

        # Summary
        if verbose:
            print(f"\n{'='*60}")
            print(f"  PIPELINE COMPLETE")
            print(f"{'='*60}")
            print(f"  {self.ticket_mgr.get_summary(ticket.ticket_id)}")

        return {
            "ticket_id": ticket.ticket_id,
            "response": response.message,
            "escalated": escalation.should_escalate,
            "sentiment": sentiment.sentiment,
            "confidence": response.confidence
        }

print("SupportPipeline defined.")

In [ ]:
# --- Full Pipeline Demo ---

support = SupportPipeline(kb=kb)

result = support.handle(
    customer_id="CUST-042",
    message="I was overcharged on my Pro Plan last month and I've been waiting "
            "for a week with no response. This is very disappointing!"
)

In [ ]:
# --- Display the result ---

print(f"Ticket ID:  {result['ticket_id']}")
print(f"Escalated:  {result['escalated']}")
print(f"Sentiment:  {result['sentiment']}")
print(f"Confidence: {result['confidence']}")
print(f"\nResponse:\n{result['response']}")

---
## 10 Domain Variants

The support agent architecture adapts across industries by customizing the knowledge base, intent taxonomy, and escalation rules.

In [ ]:
@dataclass
class SupportDomainConfig:
    """Domain-specific support configuration."""
    name: str
    intents: list[str]
    escalation_triggers: list[str]
    tone_guidelines: str
    compliance_rules: list[str]

SUPPORT_DOMAINS = {
    "saas": SupportDomainConfig(
        name="SaaS Product Support",
        intents=["billing", "technical", "account", "feature_request", "cancellation"],
        escalation_triggers=["data loss", "security breach", "compliance"],
        tone_guidelines="Friendly and helpful. Use the customer's name.",
        compliance_rules=["Never share other customers' data", "Follow GDPR for EU users"]
    ),
    "healthcare": SupportDomainConfig(
        name="Healthcare Support",
        intents=["appointment", "prescription", "insurance", "records", "emergency"],
        escalation_triggers=["emergency", "adverse reaction", "insurance denial"],
        tone_guidelines="Professional and compassionate. Never give medical advice.",
        compliance_rules=["HIPAA compliance required", "Never confirm/deny patient identity"]
    ),
    "financial": SupportDomainConfig(
        name="Financial Services Support",
        intents=["account_balance", "transaction", "fraud", "loan", "investment"],
        escalation_triggers=["fraud", "unauthorized transaction", "regulatory complaint"],
        tone_guidelines="Professional and precise. Always verify identity first.",
        compliance_rules=["PCI-DSS for card data", "Never disclose full account numbers"]
    )
}

for name, cfg in SUPPORT_DOMAINS.items():
    print(f"\n{cfg.name}")
    print(f"  Intents:    {cfg.intents}")
    print(f"  Escalation: {cfg.escalation_triggers}")
    print(f"  Compliance: {cfg.compliance_rules}")

---
## 11 Exercises

### Exercise 1 (Conceptual)

The current pipeline processes each message independently. In a real support system, the agent needs to maintain conversation context across multiple turns.

**Question:** Design a conversation memory system that:
- Tracks the full conversation history for context.
- Summarizes long conversations to fit within the LLM context window.
- Detects when the customer is repeating a question (indicating the agent is not being helpful).
- Carries over relevant entities and context between turns.

Describe the data model, summarization strategy, and how this integrates with the existing pipeline.

---

### Exercise 2 (Coding)

Implement a `SatisfactionPredictor` that estimates CSAT (Customer Satisfaction) score at the end of a conversation without explicitly asking the customer.

```python
@dataclass
class CSATEstimate:
    score: float           # 1.0 - 5.0
    confidence: float      # 0.0 - 1.0
    positive_signals: list[str]
    negative_signals: list[str]
    recommendation: str    # How to improve satisfaction

class SatisfactionPredictor:
    def predict(self, ticket: SupportTicket) -> CSATEstimate:
        # Analyze the full conversation:
        # - Sentiment trajectory (improving or worsening?)
        # - Resolution speed (number of turns)
        # - Whether the issue was actually resolved
        # - Customer's final message tone
        pass
```

---

### Exercise 3 (Design)

Design a **multilingual support system** that:
- Detects the customer's language from the first message.
- Retrieves knowledge articles in any language (cross-lingual RAG).
- Generates responses in the customer's language.
- Handles code-switching (customer mixing languages mid-conversation).

Write a design document covering:
1. Language detection and routing
2. Cross-lingual embedding strategy (multilingual models vs translate-then-embed)
3. Response generation with language consistency
4. Quality assurance for non-English responses

In [ ]:
# --- Scratch cell for exercises ---

print("Ready for exercises.")